# Исследовательский анализ данных расхода лыжного инвентаря
Анализ исторических данных для выявления паттернов сезонности и трендов

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from prophet import Prophet
from prophet.plot import plot_plotly, plot_components_plotly
import plotly.graph_objs as go

In [ ]:
# Настройки визуализации
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
# Загрузка данных из БД
import sys
sys.path.append('..')
from src.data.database import DatabaseManager

db = DatabaseManager()
df = db.get_historical_consumption(
    nomenclature_id=123,  # Лыжи гоночные
    start_date='2023-01-01'
)

print(f"Загружено {len(df)} записей")
print(f"Период: {df['ds'].min()} - {df['ds'].max()}")
df.head()

In [ ]:
# Визуализация временного ряда
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df['ds'],
    y=df['y'],
    mode='lines+markers',
    name='Выдачи инвентаря'
))
fig.update_layout(
    title='Динамика выдач лыжного инвентаря',
    xaxis_title='Дата',
    yaxis_title='Количество',
    height=500
)
fig.show()

In [ ]:
# Анализ сезонности
df['month'] = pd.to_datetime(df['ds']).dt.month
df['year'] = pd.to_datetime(df['ds']).dt.year

monthly_stats = df.groupby('month')['y'].agg(['mean', 'std', 'count']).reset_index()
monthly_stats

In [ ]:
# Визуализация сезонного профиля
fig, ax = plt.subplots(figsize=(12, 6))
months = ['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн',
          'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек']
ax.bar(months, monthly_stats['mean'].values)
ax.set_title('Среднемесячное количество выдач инвентаря')
ax.set_xlabel('Месяц')
ax.set_ylabel('Среднее количество')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Обучение модели Prophet
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    seasonality_mode='multiplicative'
)
model.add_country_holidays(country_name='RU')
model.fit(df)

In [ ]:
# Прогноз на 90 дней
future = model.make_future_dataframe(periods=90)
forecast = model.predict(future)

# Визуализация прогноза
plot_plotly(model, forecast)

In [ ]:
# Компоненты модели (тренд, сезонность)
plot_components_plotly(model, forecast)

In [ ]:
# Оценка качества модели
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# Разделение на train/test
train_size = int(len(df) * 0.8)
train_df = df[:train_size]
test_df = df[train_size:]

model_test = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    seasonality_mode='multiplicative'
)
model_test.fit(train_df)

future_test = model_test.make_future_dataframe(periods=len(test_df))
forecast_test = model_test.predict(future_test)

# Метрики
y_true = test_df['y'].values
y_pred = forecast_test['yhat'].values[-len(test_df):]

mae = mean_absolute_error(y_true, y_pred)
mape = mean_absolute_percentage_error(y_true, y_pred)

print(f"MAE: {mae:.2f}")
print(f"MAPE: {mape:.2%}")